# Stage 2 - Data Quality Report

Reads the raw source CSV and applies quality flags before Bronze ingestion.

| Flag | Rule |
|------|------|
| VALID | All checks pass |
| INVALID_CUSTOMER | customer_id IS NULL or empty |
| INVALID_QUANTITY | quantity <= 0 |
| INVALID_AMOUNT | net_amount < 0 |
| INVALID_PRODUCT | product_id = 'UNKNOWN' or NULL |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

## 1. Load source CSV

In [ ]:
SOURCE_PATH = "/Volumes/workspace/default/sales_data/sales_source_1500.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

print(f"Total records loaded: {df_raw.count()}")
df_raw.printSchema()

## 2. Apply quality flags

In [ ]:
def apply_quality_flag(df):
    """
    Assigns a single quality_flag to each row.
    Priority order: INVALID_CUSTOMER > INVALID_PRODUCT > INVALID_QUANTITY > INVALID_AMOUNT > VALID
    """
    return df.withColumn(
        "quality_flag",
        F.when(
            F.col("customer_id").isNull() | (F.trim(F.col("customer_id").cast(StringType())) == ""),
            F.lit("INVALID_CUSTOMER"),
        )
        .when(
            F.col("product_id").isNull() | (F.trim(F.col("product_id").cast(StringType())) == "UNKNOWN"),
            F.lit("INVALID_PRODUCT"),
        )
        .when(
            F.col("quantity").cast("double") <= 0,
            F.lit("INVALID_QUANTITY"),
        )
        .when(
            F.col("net_amount").cast("double") < 0,
            F.lit("INVALID_AMOUNT"),
        )
        .otherwise(F.lit("VALID")),
    )

df_flagged = apply_quality_flag(df_raw)

## 3. Quality summary

In [ ]:
quality_summary = (
    df_flagged.groupBy("quality_flag")
    .count()
    .orderBy(F.desc("count"))
)

print("=== Data Quality Summary ===")
quality_summary.show(truncate=False)

total       = df_flagged.count()
valid_count = df_flagged.filter(F.col("quality_flag") == "VALID").count()
bad_count   = total - valid_count

print(f"Total records : {total}")
print(f"Valid records : {valid_count}  ({100 * valid_count / total:.1f}%)")
print(f"Bad records   : {bad_count}   ({100 * bad_count  / total:.1f}%)")

## 4. Persist flagged staging view for downstream tasks

In [ ]:
# Write to a Delta staging table so other notebooks can read it
STAGING_TABLE = "workspace.default.capstone_staging_sales"

(
    df_flagged.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(STAGING_TABLE)
)

print(f"Staging table written: {STAGING_TABLE}")

## 5. Sample bad records

In [ ]:
print("=== Sample INVALID records ===")
(
    df_flagged
    .filter(F.col("quality_flag") != "VALID")
    .select("order_id", "customer_id", "product_id", "quantity", "net_amount", "quality_flag")
    .show(20, truncate=False)
)